Menyiapkan Dataset
-

In [51]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


Import dan membuat SprakSession
-

In [52]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas7-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap.")
print("Versi Spark:", spark.version)

SparkSession siap.
Versi Spark: 3.5.9


A. EXTRACT
-

Membaca ketiga sumber data dari HDFS

In [53]:
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv",
    header=True,
    inferSchema=True
)

df_cabang = spark.read.json(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json"
)

df_menu = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv",
    header=True,
    inferSchema=True
)

print("Jumlah transaksi :", df_transaksi.count())
print("Jumlah cabang    :", df_cabang.count())
print("Jumlah menu      :", df_menu.count())

Jumlah transaksi : 4000
Jumlah cabang    : 3
Jumlah menu      : 15


Cek struktur masing-masing DataFrame

In [54]:
print("=== Schema Transaksi ===")
df_transaksi.printSchema()

print("=== Schema Cabang ===")
df_cabang.printSchema()

print("=== Schema Menu ===")
df_menu.printSchema()

=== Schema Transaksi ===
root
 |-- trx_id: string (nullable = true)
 |-- cabang_id: integer (nullable = true)
 |-- menu_id: integer (nullable = true)
 |-- qty: integer (nullable = true)

=== Schema Cabang ===
root
 |-- cabang_id: long (nullable = true)
 |-- kepala_barista: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- nama_cabang: string (nullable = true)

=== Schema Menu ===
root
 |-- menu_id: integer (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori_menu: string (nullable = true)
 |-- harga: integer (nullable = true)



B. TRANSFORM — JOIN & KOLOM TURUNAN
-

Menggabungkan ketiga DataFrame

In [55]:
df_gabungan = df_transaksi.join(
    df_cabang,
    on="cabang_id",
    how="inner"
).join(
    df_menu,
    on="menu_id",
    how="inner"
)

df_gabungan.show(10)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|
|     14|        2|   KN5|  3|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-14|          Kopi|18000|
|      4|        3|

Menambahkan total_penjualan

In [56]:
df_gabungan = df_gabungan.withColumn(
    "total_penjualan",
    col("qty") * col("harga")
)

df_gabungan.show(10)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


In [57]:
df_gabungan.printSchema()

root
 |-- menu_id: integer (nullable = true)
 |-- cabang_id: integer (nullable = true)
 |-- trx_id: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- kepala_barista: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- nama_cabang: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori_menu: string (nullable = true)
 |-- harga: integer (nullable = true)
 |-- total_penjualan: integer (nullable = true)



C. TRANSFORM — WINDOW FUNCTION
-

Ringkas total penjualan per cabang dan menu

In [58]:
ringkasan_menu_cabang = df_gabungan.groupBy(
    "nama_cabang",
    "nama_menu"
).agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

ringkasan_menu_cabang.show()

+--------------------+---------+---------------+
|         nama_cabang|nama_menu|total_penjualan|
+--------------------+---------+---------------+
|Kopi Nusantara Se...|  Menu-10|        4806000|
|Kopi Nusantara Se...|   Menu-9|        8715000|
|Kopi Nusantara Yo...|   Menu-4|        4180000|
|Kopi Nusantara Yo...|   Menu-1|        7140000|
|Kopi Nusantara Ma...|   Menu-2|        4730000|
|Kopi Nusantara Yo...|   Menu-2|        4356000|
|Kopi Nusantara Yo...|   Menu-6|        4378000|
|Kopi Nusantara Ma...|   Menu-4|        4686000|
|Kopi Nusantara Ma...|   Menu-6|        4510000|
|Kopi Nusantara Yo...|  Menu-13|        6335000|
|Kopi Nusantara Ma...|   Menu-9|        7665000|
|Kopi Nusantara Ma...|  Menu-13|        6895000|
|Kopi Nusantara Se...|   Menu-6|        5258000|
|Kopi Nusantara Ma...|  Menu-15|        7380000|
|Kopi Nusantara Se...|  Menu-14|        3366000|
|Kopi Nusantara Yo...|   Menu-7|        3816000|
|Kopi Nusantara Ma...|   Menu-5|        4378000|
|Kopi Nusantara Se..

Membuat window function, Menggunakan row-number(), dan Menampilkan hasil top-2

In [59]:
window_cabang = Window.partitionBy(
    "nama_cabang"
).orderBy(
    col("total_penjualan").desc()
)

top2_menu = ringkasan_menu_cabang.withColumn(
    "peringkat",
    row_number().over(window_cabang)
).filter(
    col("peringkat") <= 2
)

top2_menu.orderBy(
    "nama_cabang",
    "peringkat"
).show(10)

+--------------------+---------+---------------+---------+
|         nama_cabang|nama_menu|total_penjualan|peringkat|
+--------------------+---------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|        8160000|        2|
+--------------------+---------+---------------+---------+



D. TRANSFORM — SPARK SQL
-

Buat temporary view dan jalankan SparkSQL

In [60]:
df_gabungan.createOrReplaceTempView("gabungan_tugas7")

hasil_sql = spark.sql("""
    SELECT
        kepala_barista,
        SUM(total_penjualan) AS total_penjualan
    FROM gabungan_tugas7
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
""")

hasil_sql.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



E. LOAD
-

Membuat folder processed

In [61]:
!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/processed

Menyimpan df_gabungan sebagai Parquet

In [62]:
df_gabungan.write \
    .mode("overwrite") \
    .partitionBy("kategori_menu") \
    .parquet(
        "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"
    )

print("Data gabungan berhasil disimpan sebagai Parquet.")

Data gabungan berhasil disimpan sebagai Parquet.


Verifikasi folder HDFS

In [63]:
!hdfs dfs -ls /user/mahasiswa/tugas7/processed/laporan

Found 4 items
-rw-r--r--   3 zaydann supergroup          0 2026-10-07 17:45 /user/mahasiswa/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - zaydann supergroup          0 2026-10-07 17:45 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - zaydann supergroup          0 2026-10-07 17:45 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - zaydann supergroup          0 2026-10-07 17:45 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Non-Kopi


Membaca kembali Parquet

In [64]:
df_verifikasi = spark.read.parquet(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"
)

print("Jumlah baris setelah dibaca kembali:", df_verifikasi.count())

Jumlah baris setelah dibaca kembali: 4000


In [65]:
df_verifikasi.show(10)

+-------+---------+------+---+--------------+----------+--------------------+---------+-----+---------------+--------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu|harga|total_penjualan| kategori_menu|
+-------+---------+------+---+--------------+----------+--------------------+---------+-----+---------------+--------------+
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|35000|         140000|Makanan Ringan|
|      9|        3|   KN8|  4|         Nadia|  Semarang|Kopi Nusantara Se...|   Menu-9|35000|         140000|Makanan Ringan|
|      3|        1|  KN10|  1|         Nadia|  Magelang|Kopi Nusantara Ma...|   Menu-3|30000|          30000|Makanan Ringan|
|     13|        2|  KN13|  3|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-13|35000|         105000|Makanan Ringan|
|     12|        3|  KN15|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-12|35000|          35000|Makanan Ringan|


F. RINGKASAN EKSEKUTIF
-

Membuat tabel ringkasan per cabang

In [66]:
ringkasan_cabang = df_gabungan.groupBy(
    "nama_cabang",
    "kota",
    "kepala_barista"
).agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(
    col("total_penjualan").desc()
)

ringkasan_cabang.show(truncate=False)

+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|nama_cabang              |kota      |kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Semarang  |Semarang  |Nadia         |90542000       |1393            |64997.8463747308         |
|Kopi Nusantara Magelang  |Magelang  |Nadia         |87138000       |1302            |66926.26728110599        |
|Kopi Nusantara Yogyakarta|Yogyakarta|Reza          |84645000       |1305            |64862.06896551724        |
+-------------------------+----------+--------------+---------------+----------------+-------------------------+



Berdasarkan hasil analisis, Kopi Nusantara Semarang memiliki kinerja penjualan paling tinggi dengan total penjualan sebesar Rp90.542.000. Jumlah transaksi di cabang tersebut mencapai 1.393 transaksi, sehingga menjadi cabang dengan performa penjualan terbaik dibandingkan cabang lainnya. Sementara itu, Kopi Nusantara Yogyakarta memiliki kinerja penjualan paling rendah dengan total penjualan sebesar Rp84.645.000. Dari hasil window function pada bagian C, dua menu dengan penjualan tertinggi di cabang Yogyakarta adalah Menu-12 sebesar Rp9.275.000 dan Menu-15 sebesar Rp8.160.000. Karena memiliki nilai penjualan paling tinggi, Menu-12 dapat lebih banyak dipromosikan melalui paket menu, diskon pada waktu tertentu, atau digabungkan dengan produk lainnya. Strategi tersebut diharapkan dapat meningkatkan penjualan Menu-12 sekaligus membantu meningkatkan kinerja cabang Yogyakarta.

Menutup SparkSession
-

In [69]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
